# M3L4 E11 — Golden Dataset sobre LangGraph + scores en Langfuse
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

**Objetivo:** ejecutar el golden dataset sobre el grafo LangGraph y guardar scores de correctitud en Langfuse para tener un dashboard de calidad automático.

## Flujo
```
golden_dataset
    ↓
graph.invoke() con CallbackHandler
    ↓
Capturar trace_id (langfuse_handler.last_trace_id)
    ↓
langfuse.create_score(trace_id, 'routing_correct', correct)
    ↓
Langfuse UI → Scores tab → routing_correct por trace
```

## Qué verás en Langfuse
- **Trace table:** una trace por caso del golden dataset con tag `golden-run-v1`
- **Score:** `routing_correct` (1 = correcto, 0 = incorrecto) por trace
- **Filtros:** comparar por intent, ver cuáles fallaron
- **Tendencias:** ejecutar el dataset con v1 y v2 para comparar

In [ ]:
!pip install -q langfuse langchain langchain-openai langgraph pandas
print('Instalación completa.')

In [ ]:
import os
from getpass import getpass

os.environ['LANGFUSE_PUBLIC_KEY'] = getpass('Langfuse Public Key: ')
os.environ['LANGFUSE_SECRET_KEY'] = getpass('Langfuse Secret Key: ')
os.environ['LANGFUSE_BASE_URL']   = 'https://cloud.langfuse.com'
os.environ['OPENAI_API_KEY']      = getpass('OpenAI API Key: ')
print('Credenciales OK.')

In [ ]:
import pandas as pd
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END
from langfuse import get_client
from langfuse.langchain import CallbackHandler

print('Imports OK.')

## Grafo LangGraph (reutilizado de E09)

> El grafo ya está implementado. No hay TODO en esta sección.

In [ ]:
def route_query_v2(query):
    q = query.lower()
    det = []
    if any(w in q for w in ['vacaciones','licencia','recibo','nómina','rrhh']): det.append('hr')
    if any(w in q for w in ['vpn','error','app','laptop','wifi','login','contraseña']): det.append('it')
    if any(w in q for w in ['factura','pago','reembolso','gasto','cobro','comprobante','salario']): det.append('finance')
    if any(w in q for w in ['contrato','legal','confidencialidad','nda','acuerdo']): det.append('legal')
    if len(det) > 1: return 'multi_intent'
    if len(det) == 1: return det[0]
    if len(q.split()) <= 2: return 'clarification'
    return 'general'

class AgentState(TypedDict):
    query: str
    intent: str
    response: str

def router_node(s): return {'intent': route_query_v2(s['query'])}
def hr_node(s):      return {'response': 'HRAgent: Para vacaciones o recibos, ingresá al portal de RRHH.'}
def it_node(s):      return {'response': 'ITAgent: Para VPN o errores técnicos, abrí un ticket.'}
def finance_node(s): return {'response': 'FinanceAgent: Para facturas, revisá el portal de facturación.'}
def legal_node(s):   return {'response': 'LegalAgent: Para contratos o NDAs, contactá al equipo legal.'}
def general_node(s): return {'response': 'GeneralAgent: Necesito más información.'}

def route_to_node(s):
    return {'hr':'hr_node','it':'it_node','finance':'finance_node','legal':'legal_node'}.get(s['intent'],'general_node')

builder = StateGraph(AgentState)
builder.add_node('router_node', router_node)
for name, fn in [('hr_node',hr_node),('it_node',it_node),('finance_node',finance_node),('legal_node',legal_node),('general_node',general_node)]:
    builder.add_node(name, fn)
builder.set_entry_point('router_node')
builder.add_conditional_edges('router_node', route_to_node,
    {'hr_node':'hr_node','it_node':'it_node','finance_node':'finance_node','legal_node':'legal_node','general_node':'general_node'})
for n in ['hr_node','it_node','finance_node','legal_node','general_node']:
    builder.add_edge(n, END)
graph = builder.compile()
print('Grafo listo.')

## Golden Dataset

In [ ]:
golden_dataset = [
    {'id': 'case_001', 'query': '¿Cómo solicito mis días de vacaciones?',          'expected_intent': 'hr'},
    {'id': 'case_002', 'query': 'Mi VPN no conecta desde ayer',                     'expected_intent': 'it'},
    {'id': 'case_003', 'query': 'Necesito ver mi factura del mes pasado',            'expected_intent': 'finance'},
    {'id': 'case_004', 'query': 'Necesito el contrato de confidencialidad actualizado', 'expected_intent': 'legal'},
    {'id': 'case_005', 'query': 'No puedo entrar al portal para ver mi recibo',     'expected_intent': 'hr'},
    {'id': 'case_006', 'query': 'ayuda',                                             'expected_intent': 'clarification'},
    {'id': 'case_007', 'query': '¿Cuándo se procesa el reembolso de gastos?',       'expected_intent': 'finance'},
    {'id': 'case_008', 'query': 'El sistema de login no me deja entrar',             'expected_intent': 'it'},
]
print(f'Golden dataset: {len(golden_dataset)} casos')

## TODO — Ejecutar golden dataset y scorear trazas

Completá el loop que:
1. Ejecuta cada caso sobre el grafo con Langfuse
2. Compara `actual_intent` con `expected_intent`
3. Usa `langfuse.create_score()` para registrar el score en Langfuse
4. Acumula los resultados para análisis local

In [ ]:
langfuse = get_client()
results = []

for case in golden_dataset:
    langfuse_handler = CallbackHandler()

    # TODO 1: invocar el grafo con el query del caso
    # config: callbacks=[langfuse_handler], metadata con tags=['golden-run-v1', 'm3l4'] y expected_intent
    output = None  # reemplazar

    if output is None:
        continue

    actual_intent = output.get('intent', '')
    correct = int(actual_intent == case['expected_intent'])

    # TODO 2: obtener el trace_id del handler
    trace_id = None  # pista: langfuse_handler.last_trace_id

    # TODO 3: si hay trace_id, crear score 'routing_correct'
    # langfuse.create_score(trace_id=trace_id, name='routing_correct', value=correct,
    #                       data_type='NUMERIC', comment=f"Expected {case['expected_intent']}, got {actual_intent}")
    if trace_id:
        pass  # reemplazar con create_score

    results.append({
        'id': case['id'],
        'query': case['query'][:40],
        'expected_intent': case['expected_intent'],
        'actual_intent': actual_intent,
        'correct': correct,
        'trace_id': trace_id
    })
    print(f"{case['id']}: {case['expected_intent']} → {actual_intent} {'✅' if correct else '❌'}")

print(f'\nEjecutados: {len(results)} casos')

In [ ]:
df = pd.DataFrame(results)
if len(df) > 0:
    print(f'Routing accuracy: {df["correct"].mean():.2%}')
    df

In [ ]:
assert len(results) > 0, 'No se ejecutaron casos'
assert 'correct' in results[0]
assert 'trace_id' in results[0]
accuracy = sum(r['correct'] for r in results) / len(results)
print(f'Checks E11 OK ✅ — Routing accuracy: {accuracy:.2%}')
print('Revisá los scores en Langfuse → Traces → column "routing_correct"')